# E7 Notebook 30 — G3 Day-Disjoint Open-World IDS: B0, B1, B2

**First real model-training study. No G3 results exist before you run this notebook.**

This notebook uses the **pre-registered, hash-frozen G3 experimental splits** over CIC-IDS2017's eight original TrafficLabelling CSVs. It fits a static-flow LightGBM detector (B1) against a training-prior control (B0), and evaluates the **same** trained detector at a *validation-derived* cost-sensitive threshold (B2). It runs **4 whole-day test folds × 5 seeds = 20 model checkpoints**.

### Critical controls

- All 8 original source SHA256 hashes are checked; derived data excludes only **288,602 fully empty rows** in the Thursday WebAttacks CSV. It neither modifies original data nor imputes missing labels. Nonblank unlabeled traffic aborts the run.
- Train, validation, test are **disjoint entire capture days** with test-day attack families unseen by name in training. These are exploratory **binary** held-out-day IDS tests on **uncorrected original source labels**—not multiclass zero-shot classification or proof of corrected ground truth.
- **No time windows:** raw CSV row adjacency was falsified as verified chronology by G1/G2. Features exclude Flow ID, source/destination IP and ports, raw Timestamp, Label, source-day identifiers, and row positions.
- **No UNSW-NB15 official-test access or retesting.** Results cannot be directly compared with the original SCV-IDS CIC random split.
- **Model backup is private:** the repository policy prohibits publishing fitted models publicly. Each trained model ZIP goes to an **unpublished GitHub draft release**. Its uploaded bytes are downloaded and verified *before* any held-out test evaluation. A private upload probe runs **before** loading 2.83 million source rows.
- **Public GitHub:** only test confusion counts/metrics, per-unseen-family supports and recalls, fixed config hashes, and the completed 20-run aggregate summary are uploaded. Each seed is synchronized immediately. Retries never overwrite differing original evidence.
- **Stop/resume:** published verified runs are skipped. Unpublished private model ZIP checkpoints are restored with locked metadata where available. Existing G0/G1/G2/M1 artifacts are never changed.

**Kaggle:** attach dataset \`dsshayan/dscicids\`, enable Internet, expose secret \`pushRare\`. CPU is sufficient; 20 LightGBM fits may take a substantial portion of the Kaggle runtime. The session limit is not controlled by this notebook. Do not interrupt while the private backup probe or a model upload is running.

**Scientific caveat:** this baseline is necessary reproducible infrastructure for evaluating future RACE-Net novelty, not itself a new high-impact journal result.

## 1. Pinned source, frozen G3 protocol and dependencies

No Git checkout, merge, reset, or branch updates are used. Each source file comes from one immutable GitHub commit and is checked by its Git blob SHA. The source manifest and hyperparameters were committed before G3 training.

In [ ]:
from pathlib import Path
import json,hashlib,importlib.util,subprocess,sys
from urllib.request import urlopen

REPO="itsCodeBakery/rare-attack-spatiotemporal-ids"
COMMIT="855baa3f8dc426cc838ae48c8df5b4c7da85f203"
STUDY="research/E7_CICIDS2017"
BLOBS={
 "scripts/e7_g3_openworld_baseline.py":"f882154ff9757abe66a7fe588039eb26c9758170",
 "scripts/e7_g3_archive.py":"348e54e8ad719f544ab51608d364b504e8d26744",
 "configs/G3_OPENWORLD_DAYHOLDOUT_PROTOCOL_001.json":"5ca597207bd8379860a3007b6cb7952efdde9b4f",
 "configs/G3_B012_HYPERPARAMETER_LOCK_001.json":"abd380bdbcab38ea3b33c1637457ffca44ef253e"
}
SNAPSHOT=Path("/kaggle/working/E7_G3_PINNED_SOURCE_001")
WORK=Path("/kaggle/working/E7_G3_DAYHOLDOUT_001")
WORK.mkdir(parents=True,exist_ok=True)

def blob_hash(body):
    return hashlib.sha1(b"blob "+str(len(body)).encode()+bytes([0])+body).hexdigest()

for name,digest in BLOBS.items():
    target=SNAPSHOT/name
    if target.exists():
        b=target.read_bytes()
    else:
        url=f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/{STUDY}/{name}"
        with urlopen(url,timeout=90) as stream:b=stream.read()
    assert blob_hash(b)==digest, "Code/protocol blob SHA mismatch: "+name
    if not target.exists():
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(b)
print("G3 FOUR PINNED FILES VERIFIED. Git history and older work unchanged.")

if importlib.util.find_spec("lightgbm") is None:
    subprocess.run([sys.executable,"-m","pip","install","-q","lightgbm==4.6.0"],check=True)
if importlib.util.find_spec("sklearn") is None:
    raise RuntimeError("scikit-learn missing; stop before loading source data")
import numpy as np,lightgbm as lgb,sklearn
print("RUNTIME:",sys.version.split()[0],"LightGBM",lgb.__version__,"sklearn",sklearn.__version__)

def import_from(name,p):
    spec=importlib.util.spec_from_file_location(name,p)
    module=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
training=import_from("g3_baseline",SNAPSHOT/"scripts/e7_g3_openworld_baseline.py")
archive=import_from("g3_backup",SNAPSHOT/"scripts/e7_g3_archive.py")
protocol=json.loads((SNAPSHOT/"configs/G3_OPENWORLD_DAYHOLDOUT_PROTOCOL_001.json").read_text())
lock=json.loads((SNAPSHOT/"configs/G3_B012_HYPERPARAMETER_LOCK_001.json").read_text())
training.check_protocol(protocol)
assert lock["seeds"]==list(training.SEEDS)
assert lock["boost_rounds"]==training.ROUNDS
assert lock["hyperparameters"]==training.PARAMS
assert lock["models_total"]==20 and lock["fold_count"]==4
assert lock["b2"]["threshold_grid"]["count"]==len(training.THRESHOLDS)
assert lock["b2"]["validation_cost"]=={"false_negative":5,"false_positive":1}
PROTOCOL_SHA=BLOBS["configs/G3_OPENWORLD_DAYHOLDOUT_PROTOCOL_001.json"]
print("PREREGISTRATION LOCK VERIFIED:",protocol["protocol_id"])
print("ALL source files still have to pass their original SHA256 checks.")

## 2. Critical private GitHub model-backup preflight **before** loading real data

This cell creates or reuses the unpublished GitHub draft release and uploads a tiny **non-sensitive** test ZIP. It then downloads that private asset, verifies its bytes and SHA256, and halts if the token lacks permission. No source file or model training is involved in this preflight.

In [ ]:
from kaggle_secrets import UserSecretsClient
token=UserSecretsClient().get_secret("pushRare")
assert token,"Missing Kaggle secret pushRare. Do not paste secret into notebook."
archive.api("GET",archive.ROOT,token)
RECEIPTS=WORK/"private_receipts"
RELEASE=archive.draft_release(token)
assert RELEASE.get("draft") is True,"Models must never be published in a public release."
print("PRIVATE DRAFT RELEASE AUTHENTICATED; will never publish it.")
status=archive.preflight_private_backups(token,RELEASE,RECEIPTS)
print("PRIVATE MODEL STORAGE PREFLIGHT:",status)
print("PREFLIGHT PASSED — binary GitHub private upload/download and hash verification work.")

## 3. Read-only preparation and audit of the original 8 CIC-IDS2017 CSVs

This step can take several minutes. It never writes derived flow rows to GitHub or disk; arrays remain in Kaggle RAM. Each file's SHA256, exact label distribution, fully blank row count, schema, selected features, and capture-day support must match the frozen G3 preregistration.

In [ ]:
DATA_ROOT=Path("/kaggle/input/datasets/dsshayan/dscicids")
assert DATA_ROOT.is_dir(),"Attach the existing dsshayan/dscicids Kaggle dataset."
days,features,source_provenance=training.build_all(DATA_ROOT,protocol,chunk_size=25000)
features_digest=hashlib.sha256(json.dumps(features,ensure_ascii=False).encode()).hexdigest()
assert len(features)>=65 and all(training.normalize(f) not in training.BANNED for f in features)
assert sum(len(d["y"]) for d in days.values())==2830743
SOURCE_HASHES={s["source_file"]:s["source_file_sha256"] for s in protocol["source_files"]}
print("DERIVED VALID ORIGINAL SOURCE FLOWS:",sum(len(d["y"]) for d in days.values()))
print("FLOW-ONLY FEATURE COUNT:",len(features),"| FEATURE HASH:",features_digest)
print("NO RAW IDENTIFIERS/TIME/DAY ROWS IN THE MODEL FEATURE MATRIX.")
print("BASELINE MODEL RUNS AUTHORIZED: 4 folds × 5 seeds, provisional source labels.")

## 4. Fit 20 models; save each model privately, then evaluate once and sync public metrics

- B0: train-class prevalence, decision threshold 0.5.
- B1: fixed 180-round LightGBM, decision threshold 0.5.
- B2: **same model** as B1, threshold selected on the validation day by prespecified cost \`5×FN + 1×FP\` over 201 thresholds with largest-threshold tie-break.

**Model backup happens before the held-out test is examined.** If a backup fails, subsequent training and test evaluation halt. Already verified metrics or model checkpoints can be recovered on rerun; no original result is overwritten. Per-family recall includes low-support warnings.

In [ ]:
import base64,gc,io,time
from urllib.error import HTTPError
from urllib.parse import quote
from collections import defaultdict

MODEL_DIR=WORK/"private_models"
MODEL_DIR.mkdir(parents=True,exist_ok=True)
RESULTS_DIR=WORK/"public_metrics"
RESULTS_DIR.mkdir(parents=True,exist_ok=True)
all_records=[]

def public_previous(fold_id,seed):
    name=f"G3_FOLD_{fold_id}_SEED_{seed}.metrics.json"
    url=archive.ROOT+"/contents/"+quote(archive.PUBLIC_PREFIX+"/"+name,safe="/")
    try:
        item=archive.api("GET",url,token)
    except HTTPError as e:
        if e.code==404:return None
        raise
    record=json.loads(base64.b64decode(item["content"]))
    archive.safe_metrics(record)
    return record

for fold in protocol["folds"]:
    fold_id=int(fold["fold"].split("_")[-1])
    print("\n"+"="*75)
    print("DAY-HOLDOUT",fold["fold"],"train",fold["train_days"],
          "validation",fold["validation_day"],"TEST",fold["test_day"],flush=True)
    # Build split only for folds with missing scores.
    missing=[seed for seed in training.SEEDS if public_previous(fold_id,seed) is None]
    if not missing:
        for seed in training.SEEDS:
            old=public_previous(fold_id,seed)
            assert old["protocol_blob_sha"]==PROTOCOL_SHA
            assert old["features_sha256"]==features_digest
            assert old["source_hashes"]==SOURCE_HASHES
            assert old["test_day"]==fold["test_day"]
            all_records.append(old)
        print("ALL 5 ORIGINAL SEEDS ALREADY VERIFIED ON GITHUB — skipped fitting.")
        continue

    TX,TY,VX,VY,EX,EY,test_labels=training.get_fold(days,fold)
    print("TRAIN / VAL / TEST ROWS:",len(TY),len(VY),len(EY))
    train_rate=float(np.mean(TY))
    for seed in training.SEEDS:
        old=public_previous(fold_id,seed)
        if old is not None:
            assert old["protocol_blob_sha"]==PROTOCOL_SHA
            assert old["features_sha256"]==features_digest
            assert old["source_hashes"]==SOURCE_HASHES
            all_records.append(old)
            print("G3",fold_id,"SEED",seed,"GITHUB VERIFIED METRICS ALREADY EXIST — skipped.")
            continue

        name=f"G3_FOLD_{fold_id}_SEED_{seed}_PRIVATE.zip"
        local=MODEL_DIR/name
        required={"schema":"E7_G3_PRIVATE_MODEL_001",
          "fold":fold["fold"],"seed":seed,
          "protocol_blob_sha":PROTOCOL_SHA,"features_sha256":features_digest,
          "train_days":fold["train_days"],"validation_day":fold["validation_day"],
          "test_day":fold["test_day"],"feature_names":features}
        known_assets=archive.assets(RELEASE,token)

        if local.exists():
            private_bytes=local.read_bytes()
            booster,metadata=archive.unpack_model(private_bytes,required)
            print("RECOVERED ORIGINAL LOCAL PRIVATE MODEL CHECKPOINT.",flush=True)
        elif name in known_assets:
            private_bytes=archive.get_private_asset(known_assets[name],token)
            booster,metadata=archive.unpack_model(private_bytes,required)
            local.write_bytes(private_bytes)
            print("RESTORED EXISTING PRIVATE MODEL FROM DRAFT RELEASE.",flush=True)
        else:
            print("TRAINING FOLD",fold_id,"SEED",seed,
                  "model",training.ROUNDS,"rounds (CPU)...",flush=True)
            booster=training.train_seed(TX,TY,features,seed)
            val_pred=booster.predict(VX,num_threads=4)
            selection=training.choose_threshold(VY,val_pred)
            metadata={**required,
                "hyperparameters":{**training.PARAMS,"num_boost_round":training.ROUNDS},
                "threshold_selection":selection,
                "train_attack_rate":train_rate,"training_rows":int(len(TY)),
                "lightgbm_version":lgb.__version__}
            private_bytes=archive.pack_model(booster,metadata)
            tmp=local.with_suffix(".zip.writing")
            tmp.write_bytes(private_bytes)
            tmp.replace(local)
            print("PRIVATE MODEL SAVED LOCALLY. ORIGINAL CHECKPOINT WILL NEVER BE OVERWRITTEN.",flush=True)

        assert metadata["train_attack_rate"]==train_rate
        assert metadata["training_rows"]==len(TY)
        assert metadata["lightgbm_version"]==lgb.__version__,(
            "LightGBM version changed after original checkpoint; stop.")
        # Confirm restored private model's threshold matches EXACT current held-out validation.
        vpred=booster.predict(VX,num_threads=4)
        check_selection=training.choose_threshold(VY,vpred)
        assert check_selection==metadata["threshold_selection"],(
            "Restored training/validation model calibration mismatch. No test access.")
        archive_status=archive.archive_model(private_bytes,token=token,
            release=RELEASE,asset_name=name,receipt_dir=RECEIPTS)
        assert archive_status.startswith("PRIVATE_")
        print("PRIVATE GITHUB DRAFT MODEL BACKUP VERIFIED:",archive_status,flush=True)

        # No test prediction until model and validation settings have been frozen
        # and private artifact independently fetched+SHA checked.
        test_probability=booster.predict(EX,num_threads=4)
        b0=training.evaluate(EY,np.full(len(EY),train_rate),0.5,test_labels)
        b1=training.evaluate(EY,test_probability,0.5,test_labels)
        b2=training.evaluate(EY,test_probability,
                              metadata["threshold_selection"]["threshold"],test_labels)
        public={
          "schema":"E7_G3_PUBLIC_METRICS_001",
          "experiment":protocol["protocol_id"],
          "protocol_blob_sha":PROTOCOL_SHA,
          "fold":fold["fold"],"seed":seed,
          "train_days":fold["train_days"],
          "validation_day":fold["validation_day"],"test_day":fold["test_day"],
          "source_hashes":SOURCE_HASHES,"features_sha256":features_digest,
          "hyperparameters":{**training.PARAMS,"num_boost_round":training.ROUNDS},
          "training_rows":int(len(TY)),"validation_rows":int(len(VY)),
          "test_rows":int(len(EY)),"train_attack_rate":train_rate,
          "validation_selection":metadata["threshold_selection"],
          "b0":b0,"b1":b1,"b2":b2,
          "model_asset_sha256":archive.sha(private_bytes),
          "private_backup_verified":True,
          "original_label_provenance":"UNVERIFIED_ORIGINAL_CICIDS2017_LABELS",
          "limitations":[
            "Original CICIDS2017 labels not independently corrected or publisher-authenticated",
            "Heldout capture-day attacks are unseen by name, confounded with capture-day shift",
            "Static flow features only; no verified temporal causality or source row context",
            "Do not compare numerically with original SCV-IDS CIC random-split result",
            "Frozen UNSW official test was never opened or rerun",
            "Small attack support (Heartbleed/Infiltration/SQL Injection) limits inferences"
          ]
        }
        state=archive.publish_metrics(public,token=token,output_folder=RESULTS_DIR)
        all_records.append(public)
        print("GITHUB PUBLIC METRICS:",state,
              "| G3",fold_id,"SEED",seed,
              "| B1 Macro-F1",round(b1["macro_f1"],4),
              "| B2 Macro-F1",round(b2["macro_f1"],4),
              "| B2 FPR",round(b2["fpr"],4),flush=True)
        del test_probability,vpred,booster
        gc.collect()
    del TX,TY,VX,VY,EX,EY,test_labels
    gc.collect()

print("\nG3 REPRODUCIBLE BASELINE COMPLETED:",len(all_records),"/20 model seeds")
assert len(all_records)==20
summary_by={}
for fold in protocol["folds"]:
    block=[r for r in all_records if r["fold"]==fold["fold"]]
    assert sorted(r["seed"] for r in block)==list(training.SEEDS)
    def avg(field,metric):
        return float(np.mean([r[field][metric] for r in block]))
    def std(field,metric):
        return float(np.std([r[field][metric] for r in block],ddof=1))
    summary_by[fold["fold"]]={
        "test_day":fold["test_day"],"n_seeds":5,
        "b0_macro_f1_mean":avg("b0","macro_f1"),
        "b1_macro_f1_mean":avg("b1","macro_f1"),
        "b1_macro_f1_std":std("b1","macro_f1"),
        "b2_macro_f1_mean":avg("b2","macro_f1"),
        "b2_macro_f1_std":std("b2","macro_f1"),
        "b2_attack_recall_mean":avg("b2","recall_attack"),
        "b2_fpr_mean":avg("b2","fpr")}
summary={
  "schema":"E7_G3_SEED_SUMMARY_001",
  "experiment":protocol["protocol_id"],
  "protocol_blob_sha":PROTOCOL_SHA,
  "model_count":20,"fold_count":4,"seed_count":5,
  "fold_summary":summary_by,
  "limitations":[
     "Original CICIDS2017 labels provisional; corrected-source sensitivity pending",
     "Day-specific attack families and capture settings confounded; not independent real-world zero-shot proof",
     "No verified related-flow chronology or spatiotemporal input",
     "Not directly comparable with the base SCV-IDS CIC random split or frozen UNSW test"
  ]}
final_status=archive.publish_summary(summary,token=token,output_folder=RESULTS_DIR)
print("GITHUB SUMMARY:",final_status)
print("PUBLIC STUDY FOLDER: https://github.com/"+REPO+"/tree/main/"+archive.PUBLIC_PREFIX)
print("ALL 20 PRIVATE MODELS VERIFIED IN UNPUBLISHED GITHUB DRAFT RELEASE.")
print("Send the printed per-fold metrics and GitHub URL to ChatGPT. No model reruns yet.")